# A3 · corrección telúrica — notebook de análisis (`debug`)

**Objeto:** ROXs12b  |  **Run:** `ROXs12b_realigned`  |  **Spec:** [`docs/spec_A3_v2_codex_telluric.md`](../../../docs/spec_A3_v2_codex_telluric.md)

Este notebook **no llama a la cadena**: rehace la decisión de A3 aquí dentro, con el código a la vista, para que puedas **probar, cambiar y ajustar sin tocar `musepipe`**. El notebook de auditoría equivalente es [`../A3_telluric.ipynb`](../A3_telluric.ipynb), que sí lee el QC de la etapa.

Cómo está montado, y por qué:

1. **Perillas** arriba del todo, con el valor que usa la cadena para este run.
2. **Las funciones numéricas, copiadas literalmente** de `musepipe`. Se copian (en vez de importarse) para que puedas editarlas: todo lo que viene después usa estos nombres locales.
3. **Chequeo de deriva** — avisa si `musepipe` cambió y esta copia se quedó atrás.
4. El proceso **paso a paso**, cada uno con su diagnóstico.
5. **Comparación con el producto de la cadena**: con las perillas por defecto debe salir *idéntico*; en cuanto cambias algo, te dice qué se movió y dónde.

### La pregunta que este notebook viene a cerrar

**Por qué la profundidad telúrica de este objeto mide lo que mide**, y si ese número aguanta como base de una decisión. Mientras no se explique, la corrección telúrica de la cadena no es citable ni como «medida y descartada» ni como «necesaria»: lo único medido es lo que el estimador anotó en *ese* cubo.

Se contrastan cuatro hipótesis (§7, §8, §9, §10), cada una con su número — y las cifras las imprimen las celdas a partir del QC y del cubo, no este texto:

1. el DRS ya quitó las bandas, exposición a exposición, y lo que queda es **residuo** y no absorción;
2. combinar exposiciones a masas de aire distintas **difumina** la banda;
3. medir con la **mediana sobre toda la banda** diluye un núcleo estrecho;
4. el **continuo curvado** hace que la cuerda entre los dos laterales fabrique profundidad que no es absorción — medido con ventanas de control y con su control positivo, que es lo que permite distinguir «no hay sesgo» de «la prueba está ciega».

> Lo que NO se copia: la resolución del cubo de entrada, la comprobación del entorno de `molecfit` y el escritor del QC — son E/S de la etapa, no su matemática. Y lo que este notebook **no** repite del de auditoría: el porqué de `STD_TELLURIC` frente a `molecfit`, el volcado campo a campo de los dos esquemas de QC y la figura de la curva de transmisión.


In [ ]:
import json, sys
from pathlib import Path

import numpy as np
from astropy.io import fits
import matplotlib.pyplot as plt

# Resolución de las figuras EN PANTALLA. `savefig` guarda a 300 dpi, pero
# lo que se ve dentro del notebook lo fija el backend inline, que va a 100
# dpi por defecto y sale borroso. `retina` dobla los píxeles sin cambiar el
# tamaño aparente; fuera de IPython no hace nada y queda el rcParam.
import matplotlib as mpl
mpl.rcParams['figure.dpi'] = 120
mpl.rcParams['savefig.dpi'] = 200
try:
    from matplotlib_inline.backend_inline import set_matplotlib_formats
    set_matplotlib_formats('retina')
except Exception:
    pass
_here = Path.cwd()
ROOT = next(p for p in (_here, *_here.parents) if (p / 'musepipe').is_dir())
sys.path.insert(0, str(ROOT)); sys.path.insert(0, str(ROOT / 'notebooks'))
import _nbcommon as nb

RUN_ID = nb.resolve_run_id('ROXs12b_realigned')
RD = nb.run_dir(RUN_ID); SD = RD / 'stages'
CFG = json.loads((RD / 'config' / 'config.json').read_text(encoding='utf-8'))['config']
# El objeto se DERIVA del run (cadena declarada en su config), no se
# escribe: un literal aquí haría que un objeto nuevo heredase el nombre
# del primero, que es lo que vigila tests/test_no_hardcoded_target.py.
TARGET = nb.run_target(RUN_ID) or nb.display_name(RUN_ID)

def resolver(p):
    """Ruta declarada en un QC -> ruta usable.

    Los QC de A3 guardan unas rutas absolutas y otras relativas a la RAÍZ
    del repo, y el cwd de un notebook es su propia carpeta. Anclar aquí
    evita el fallo que rompió el A3 de auditoría (traspaso 07-29 §6.1).
    """
    if not p:
        return None
    q = Path(p)
    return q if q.is_absolute() else (ROOT / q)

print('objeto :', TARGET, '·', nb.display_name(RUN_ID))
print('run    :', RUN_ID)
print('stages :', SD)


## 1 · Perillas

Salen del **config resuelto de la etapa**, no del `config.json` crudo. A3 es la etapa donde esto más importa: es anterior a la convención `musepipe.stages`, así que la mitad de sus perillas vive en los *defaults de argparse* y la otra mitad como literales dentro de las funciones numéricas (el umbral del 3 %, las bandas laterales del continuo, los bordes de las bandas). `stage00t_config_from_run` las reúne en un sitio; copiarlas a mano es exactamente cómo se consigue un notebook que no reproduce la cadena.

Dos avisos que la celda imprime y conviene leer:

- **ningún fichero de config las contiene**: hoy todas salen del default de la etapa, así que un cambio de default cambia lo que reproduce este notebook;
- `a3_science_needs_red_continuum` vale `True` aquí y `False` en el `store_true` de `main`. Los tres QC en disco registran `true`, y `decide_telluric` cortocircuita a `not_needed_science` si es falso: la bandera se pasó. El resolutor reproduce lo que corrió, no lo que el default sugiere.


In [ ]:
from musepipe.reduction.telluric import stage00t_config_from_run

# `project_root=ROOT` no es opcional: musepipe resuelve rutas contra el cwd, y
# el cwd de un notebook es su propia carpeta, no la raíz del repo.
X00T = stage00t_config_from_run(RUN_ID, project_root=ROOT)   # run + defaults de la etapa
RADIUS_PX      = float(X00T['a3_radius_px'])
THRESHOLD_PCT  = float(X00T['a3_threshold_pct'])
NEEDS_RED_CONT = bool(X00T['a3_science_needs_red_continuum'])
BANDS_A        = {k: tuple(v) for k, v in X00T['a3_bands_A'].items()}
PROTECTED_A    = [tuple(w) for w in X00T['a3_protected_windows_A']]
SIDE_WIDTH_A   = float(X00T['a3_side_width_A'])
GAP_A          = float(X00T['a3_gap_A'])

# O₂ A, la más profunda del rango de MUSE y el discriminante de §7. Este
# notebook la midió cuando la etapa NO la medía; desde 2026-07-31 está en
# `TELLURIC_BANDS`, así que sale del config resuelto como las demás — nunca
# como literal, que es lo que hace que un notebook deje de reproducir la cadena.
O2_A_BAND  = BANDS_A['O2_A']
BANDAS_MAS = dict(BANDS_A)   # se conserva el nombre: lo usan §5, §6, §9 y §10

# ---- a partir de aquí, cambia lo que quieras probar ----

_del_run = [k for k in X00T if k.startswith('a3_') and k in CFG]
for _k, _v in {'radio apertura (px)': RADIUS_PX, 'umbral (%)': THRESHOLD_PCT,
               'continuo rojo necesario': NEEDS_RED_CONT,
               'bandas': list(BANDS_A), 'protegidas': PROTECTED_A,
               'continuo lateral/hueco (Å)': (SIDE_WIDTH_A, GAP_A)}.items():
    print(f'  {_k:26s} {_v}')
print('\n  declaradas en el config del run:', _del_run or 'ninguna — todas son default de la etapa')


## 2 · Entradas — las reducciones del objeto y sus QC

El mismo objeto se ha reducido tres veces, y **cada reducción tiene su propio QC de A3 con su propio veredicto**. Esa es la primera mitad de la respuesta: la pregunta «¿cuánto vale la banda?» no tiene un número, tiene tres.

Los QC hermanos se **descubren**, no se nombran: se buscan los `stage00t*.json` de todos los runs del mismo objeto y se indexan por el cubo que cada uno declara. Pedirlos por nombre a `nb.load_qc` sería peor que inútil — si el fichero no existe en el run activo, la resolución cae al alias del registro y devuelve **el QC de otra reducción bajo el nombre del que pediste**, en silencio.

La celda dice también qué falta en disco: el cubo de entrada de la primera reducción **está borrado** (§10 lo reconstruye) y el «después» de la segunda vive **fuera** de su run.


In [ ]:
def qcs_de_a3():
    """Los QC de A3 de todos los runs de ESTE objeto, por su propio cubo."""
    out = {}
    for p in sorted((ROOT / 'runs').glob('*/stages/stage00t*.json')):
        run = p.parent.parent.name
        try:
            mismo = nb.run_target(run) == nb.run_target(RUN_ID)
        except Exception:
            mismo = False
        if not mismo:
            continue
        out[f'{run}/{p.name}'] = json.loads(p.read_text(encoding='utf-8'))
    return out

QC_A3 = qcs_de_a3()
# El de la cadena activa sí sale por la vía normal: así `chain.stage_runs['A3']`
# sigue mandando si algún día este objeto separa A3 en otro run.
QC_CAN = nb.load_qc_optional('stages/stage00t_qc.json', RUN_ID)

REDUCCIONES = []
for etiqueta, qc in QC_A3.items():
    dec = qc.get('decision', {}); ent = qc.get('input', {}); pro = qc.get('products', {})
    REDUCCIONES.append(dict(
        etiqueta=etiqueta,
        canonica=(QC_CAN is not None and qc.get('input', {}).get('sha256') == QC_CAN.get('input', {}).get('sha256')
                  and dec.get('verdict') == QC_CAN.get('decision', {}).get('verdict')),
        qc=qc,
        pre=resolver(ent.get('cube')),
        post=resolver(pro.get('cube_telcorr')),
        curva=resolver(pro.get('transmission')),
        yx=tuple(ent['primary_yx']) if ent.get('primary_yx') else None,
        radio=float(ent['aperture_radius_px']) if ent.get('aperture_radius_px') else None,
        aplicado=bool(dec.get('telluric_applied')),
        # `telluric_applied` es la DECISIÓN, no el hecho: con veredicto
        # `needed` sale True con el cubo todavía sin tocar. Lo que dice si
        # se aplicó de verdad es `applied_to_cube` (o, en los QC antiguos
        # que no lo traen, que `products.cube_telcorr` no esté vacío).
        aplicado_de_verdad=bool(dec.get('applied_to_cube', bool(pro.get('cube_telcorr')))),
        veredicto=dec.get('verdict'),
        profundidades=dec.get('depth_pct_by_band', {}),
    ))
REDUCCIONES.sort(key=lambda r: (r['canonica'], r['etiqueta']))

def _existe(p):
    return 'sí' if (p is not None and p.exists()) else ('BORRADO/ausente' if p else '—')

for r in REDUCCIONES:
    print(('· CANÓNICA  ' if r['canonica'] else '· histórica ') + r['etiqueta'])
    _dec = 'decidida' if r['aplicado'] else 'no hace falta'
    _hec = 'APLICADA al cubo' if r['aplicado_de_verdad'] else 'sin aplicar'
    print(f"    veredicto {r['veredicto']}  ·  corrección {_dec}, {_hec}  "
          f"apertura={r['yx']} r={r['radio']}")
    print('    profundidades declaradas:', {k: round(v, 4) for k, v in r['profundidades'].items()})
    print(f"    cubo PRE   {_existe(r['pre']):16s} {r['pre']}")
    print(f"    cubo POST  {_existe(r['post']):16s} {r['post']}")
    print(f"    curva      {_existe(r['curva']):16s} {r['curva']}")

if QC_CAN is not None and not any(r['canonica'] for r in REDUCCIONES):
    print('\nAVISO: el QC de la cadena activa no se ha emparejado con ningún hermano.')
if not REDUCCIONES:
    print('A3 no ha corrido para este objeto: las secciones siguientes lo dirán y no inventarán nada.')


## 3 · Las funciones numéricas, copiadas de `musepipe`

Copia **literal** del fuente, para que puedas editarla. Todo lo que viene después usa estos nombres locales, así que un cambio aquí se propaga al resultado — y la comparación del final lo cuantifica.

- `VerificationError` — de `musepipe/reduction/verify.py`
- `circular_aperture_mask` — de `musepipe/reduction/verify.py`
- `extract_aperture_spectrum` — de `musepipe/reduction/verify.py`
- `TelluricError` — de `musepipe/reduction/telluric.py`
- `TelluricDecision` — de `musepipe/reduction/telluric.py`
- `wavelength_axis_from_header` — de `musepipe/reduction/telluric.py`
- `window_mask` — de `musepipe/reduction/telluric.py`
- `protected_mask` — de `musepipe/reduction/telluric.py`
- `local_continuum_linear` — de `musepipe/reduction/telluric.py`
- `measure_telluric_depths` — de `musepipe/reduction/telluric.py`
- `decide_telluric` — de `musepipe/reduction/telluric.py`
- `enforce_protected_transmission` — de `musepipe/reduction/telluric.py`
- `validate_transmission_physical` — de `musepipe/reduction/telluric.py`
- `apply_transmission_to_arrays` — de `musepipe/reduction/telluric.py`
- `verify_outside_bands_unchanged` — de `musepipe/reduction/telluric.py`


In [ ]:
# ------------------------------------------------------------------
# COPIA EDITABLE. Fuente: musepipe (ver el chequeo de deriva abajo).
# ------------------------------------------------------------------
from astropy.io import fits
from dataclasses import dataclass
from typing import Callable
from typing import Mapping
from typing import Sequence
import numpy as np
# Las dos excepciones viajan porque las levantan las funciones copiadas:
# un `raise` a una clase ausente reventaría el notebook al primer hueco.

HALPHA_PROTECTED = (6540.0, 6590.0)
NALGS_PROTECTED = (5780.0, 6050.0)
PROTECTED_WINDOWS = (HALPHA_PROTECTED, NALGS_PROTECTED)
TELLURIC_BANDS = {
    "O2_B": (6864.0, 6960.0),
    "O2_A": (7590.0, 7700.0),
    "H2O_7200": (7160.0, 7340.0),
    "H2O_8200": (8130.0, 8350.0),
}
DEFAULT_FIT_REGIONS = tuple(TELLURIC_BANDS.values())


class VerificationError(RuntimeError):
    """Raised when a requested verification cannot be completed."""


def circular_aperture_mask(shape: tuple[int, int], yx: tuple[float, float], radius: float) -> np.ndarray:
    y, x = np.indices(shape, dtype=np.float64)
    cy, cx = yx
    return (y - cy) ** 2 + (x - cx) ** 2 <= radius**2


def extract_aperture_spectrum(cube: np.ndarray, yx: tuple[float, float], radius: float) -> np.ndarray:
    mask = circular_aperture_mask(cube.shape[1:], yx, radius)
    if not mask.any():
        raise VerificationError("Aperture contains no pixels.")
    return np.nansum(cube[:, mask], axis=1)


class TelluricError(RuntimeError):
    """Raised when A3 must stop at a gate or checkpoint."""


@dataclass(frozen=True)
class TelluricDecision:
    depth_pct_by_band: dict[str, float]
    telluric_applied: bool
    science_needs_red_continuum: bool
    decision: str
    checkpoint_required: bool


def wavelength_axis_from_header(header: fits.Header, n_wave: int) -> np.ndarray:
    if all(key in header for key in ("CRVAL3", "CDELT3")):
        crpix = float(header.get("CRPIX3", 1.0))
        return float(header["CRVAL3"]) + (
            np.arange(int(n_wave), dtype=np.float64) + 1.0 - crpix
        ) * float(header["CDELT3"])
    if all(key in header for key in ("CRVAL3", "CD3_3")):
        crpix = float(header.get("CRPIX3", 1.0))
        return float(header["CRVAL3"]) + (
            np.arange(int(n_wave), dtype=np.float64) + 1.0 - crpix
        ) * float(header["CD3_3"])
    raise TelluricError("Could not recover wavelength axis from DATA header.")


def window_mask(wave: Sequence[float], window: tuple[float, float]) -> np.ndarray:
    wave_arr = np.asarray(wave, dtype=np.float64)
    lo, hi = window
    return (wave_arr >= lo) & (wave_arr <= hi) & np.isfinite(wave_arr)


def protected_mask(wave: Sequence[float], windows: Sequence[tuple[float, float]] = PROTECTED_WINDOWS) -> np.ndarray:
    mask = np.zeros(np.asarray(wave).shape, dtype=bool)
    for window in windows:
        mask |= window_mask(wave, window)
    return mask


def local_continuum_linear(
    wave: Sequence[float],
    spectrum: Sequence[float],
    band: tuple[float, float],
    *,
    side_width_A: float = 40.0,
    gap_A: float = 10.0,
) -> np.ndarray:
    """Interpolate a local continuum across one telluric band."""

    wave_arr = np.asarray(wave, dtype=np.float64)
    spec = np.asarray(spectrum, dtype=np.float64)
    lo, hi = band
    left = (wave_arr >= lo - gap_A - side_width_A) & (wave_arr <= lo - gap_A)
    right = (wave_arr >= hi + gap_A) & (wave_arr <= hi + gap_A + side_width_A)
    left &= np.isfinite(spec)
    right &= np.isfinite(spec)
    if not left.any() or not right.any():
        finite = np.isfinite(spec)
        fallback = float(np.nanmedian(spec[finite])) if finite.any() else 1.0
        return np.full(wave_arr.shape, fallback, dtype=np.float64)
    x = np.array([np.nanmedian(wave_arr[left]), np.nanmedian(wave_arr[right])], dtype=np.float64)
    y = np.array([np.nanmedian(spec[left]), np.nanmedian(spec[right])], dtype=np.float64)
    if not np.all(np.isfinite(y)) or x[0] == x[1]:
        return np.full(wave_arr.shape, float(np.nanmedian(spec[np.isfinite(spec)])), dtype=np.float64)
    return np.interp(wave_arr, x, y)


def measure_telluric_depths(
    wave: Sequence[float],
    spectrum: Sequence[float],
    *,
    bands: Mapping[str, tuple[float, float]] = TELLURIC_BANDS,
    continuum: Callable[..., np.ndarray] | None = None,
    side_width_A: float = 40.0,
    gap_A: float = 10.0,
    clip_negative: bool = True,
) -> dict[str, float]:
    """Measure telluric depth as percent drop relative to local continuum.

    The defaults reproduce the frozen stage behaviour bit for bit: `continuum=None`
    means `local_continuum_linear` with its own defaults, and `clip_negative=True`
    keeps the `max(0.0, ...)` floor. The keywords exist so a *diagnostic* can vary
    what the stage keeps fixed, which until now was impossible from above — the
    sideband knobs resolved by `stage00t_config_from_run` could not reach this
    function at all, so the A3 analysis notebook could only redraw the continuum it
    plotted, never the number it compared. Nothing in the chain passes them; see
    `docs/spec_A3_v2_codex_telluric.md` §1.5.

    `clip_negative=False` matters for a band that is *already corrected*: the floor
    turns a negative depth into 0.0, so the sign — the evidence that the DRS
    over-shot rather than under-shot — is destroyed before any caller can see it.
    """

    wave_arr = np.asarray(wave, dtype=np.float64)
    spec = np.asarray(spectrum, dtype=np.float64)
    depths: dict[str, float] = {}
    for name, band in bands.items():
        mask = window_mask(wave_arr, band)
        if not mask.any():
            depths[name] = float("nan")
            continue
        if continuum is None:
            model = local_continuum_linear(
                wave_arr, spec, band, side_width_A=side_width_A, gap_A=gap_A
            )
        else:
            model = np.asarray(continuum(wave_arr, spec, band), dtype=np.float64)
        valid = mask & np.isfinite(spec) & np.isfinite(model) & (model != 0)
        if not valid.any():
            depths[name] = float("nan")
            continue
        ratio = spec[valid] / model[valid]
        depth = 100.0 * (1.0 - float(np.nanmedian(ratio)))
        depths[name] = max(0.0, depth) if clip_negative else depth
    return depths


def decide_telluric(
    depth_pct_by_band: Mapping[str, float],
    *,
    science_needs_red_continuum: bool,
    threshold_pct: float = 3.0,
) -> TelluricDecision:
    depths = {str(key): float(value) for key, value in depth_pct_by_band.items()}
    finite_depths = [value for value in depths.values() if np.isfinite(value)]
    max_depth = max(finite_depths) if finite_depths else float("nan")
    if not science_needs_red_continuum:
        decision = "not_needed_science"
        applied = False
        checkpoint = False
    elif np.isfinite(max_depth) and max_depth < threshold_pct:
        decision = "not_needed_shallow"
        applied = False
        checkpoint = False
    elif np.isfinite(max_depth):
        decision = "needed"
        applied = True
        checkpoint = True
    else:
        decision = "depth_unknown_checkpoint"
        applied = False
        checkpoint = True
    return TelluricDecision(
        depth_pct_by_band=depths,
        telluric_applied=applied,
        science_needs_red_continuum=bool(science_needs_red_continuum),
        decision=decision,
        checkpoint_required=checkpoint,
    )


def enforce_protected_transmission(
    wave: Sequence[float],
    transmission: Sequence[float],
    *,
    windows: Sequence[tuple[float, float]] = PROTECTED_WINDOWS,
) -> np.ndarray:
    trans = np.asarray(transmission, dtype=np.float64).copy()
    if trans.ndim != 1:
        raise TelluricError("Transmission must be a 1D array.")
    mask = protected_mask(wave, windows)
    if mask.shape != trans.shape:
        raise TelluricError("Transmission and wavelength arrays must have the same shape.")
    trans[mask] = 1.0
    return trans


def validate_transmission_physical(transmission: Sequence[float]) -> bool:
    trans = np.asarray(transmission, dtype=np.float64)
    return bool(np.all(np.isfinite(trans)) and np.nanmin(trans) > 0.0 and np.nanmax(trans) <= 1.0)


def apply_transmission_to_arrays(
    data: np.ndarray,
    stat: np.ndarray,
    wave: Sequence[float],
    transmission: Sequence[float],
    *,
    min_transmission: float = 0.05,
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """Apply DATA/T and STAT/T^2, forcing protected windows to T=1."""

    cube = np.asarray(data, dtype=np.float64)
    variance = np.asarray(stat, dtype=np.float64)
    if cube.shape != variance.shape:
        raise TelluricError(f"DATA shape {cube.shape} != STAT shape {variance.shape}.")
    trans = enforce_protected_transmission(wave, transmission)
    if trans.shape[0] != cube.shape[0]:
        raise TelluricError("Transmission length does not match cube wavelength axis.")
    if np.nanmin(trans) < min_transmission:
        raise TelluricError(f"Transmission below safety floor {min_transmission}.")
    scale = trans[:, None, None]
    return cube / scale, variance / (scale**2), trans


def verify_outside_bands_unchanged(
    pre_spec: Sequence[float],
    post_spec: Sequence[float],
    wave: Sequence[float],
    *,
    max_change_pct: float = 0.2,
    corrected_bands: Sequence[tuple[float, float]] = DEFAULT_FIT_REGIONS,
) -> bool:
    pre = np.asarray(pre_spec, dtype=np.float64)
    post = np.asarray(post_spec, dtype=np.float64)
    wave_arr = np.asarray(wave, dtype=np.float64)
    mask = np.isfinite(pre) & np.isfinite(post) & (pre != 0)
    for band in corrected_bands:
        mask &= ~window_mask(wave_arr, band)
    mask &= ~protected_mask(wave_arr)
    if not mask.any():
        raise TelluricError("No outside-band channels available for verification.")
    change_pct = 100.0 * np.nanmedian(np.abs(post[mask] / pre[mask] - 1.0))
    return bool(change_pct <= max_change_pct)


## 4 · Chequeo de deriva

Compara el fuente copiado arriba con el que **hoy** tiene `musepipe`. Si alguien cambió la cadena, esta celda lo dice nombrando la función: es lo que evita que este notebook siga dando resultados «de la cadena» cuando ya no lo son.


In [ ]:
import ast as _ast, hashlib as _hashlib

_SHAS = {
    "musepipe/reduction/verify.py:VerificationError": "d9acb4457343",
    "musepipe/reduction/verify.py:circular_aperture_mask": "b08d990cd3a2",
    "musepipe/reduction/verify.py:extract_aperture_spectrum": "e46a9616dd50",
    "musepipe/reduction/telluric.py:TelluricError": "5347f3394587",
    "musepipe/reduction/telluric.py:TelluricDecision": "854cf1dd3710",
    "musepipe/reduction/telluric.py:wavelength_axis_from_header": "f463214b3a31",
    "musepipe/reduction/telluric.py:window_mask": "f5bf804839d4",
    "musepipe/reduction/telluric.py:protected_mask": "3bd25d021885",
    "musepipe/reduction/telluric.py:local_continuum_linear": "342caee26693",
    "musepipe/reduction/telluric.py:measure_telluric_depths": "d9d4f4bc549d",
    "musepipe/reduction/telluric.py:decide_telluric": "bd9e31d10459",
    "musepipe/reduction/telluric.py:enforce_protected_transmission": "f904d9698e66",
    "musepipe/reduction/telluric.py:validate_transmission_physical": "c1e41df83a9e",
    "musepipe/reduction/telluric.py:apply_transmission_to_arrays": "44fedc1b757c",
    "musepipe/reduction/telluric.py:verify_outside_bands_unchanged": "a84b16e93d26",
    "musepipe/reduction/telluric.py:HALPHA_PROTECTED": "b507bce76f01",
    "musepipe/reduction/telluric.py:NALGS_PROTECTED": "9e28e71df15b",
    "musepipe/reduction/telluric.py:PROTECTED_WINDOWS": "e9651e284031",
    "musepipe/reduction/telluric.py:TELLURIC_BANDS": "e3cbad576db7",
    "musepipe/reduction/telluric.py:DEFAULT_FIT_REGIONS": "d17fc1b870b0"
}

def _pieza(cuerpo, name):
    """El nodo que define `name`: def/class, o la asignación de una constante.

    Las constantes también se vigilan: viajan copiadas igual que las
    funciones, y hasta ahora nadie comprobaba que siguieran siendo las de
    `musepipe` — añadir una banda a un diccionario dejaba esta copia atrás
    sin que nada lo dijera.
    """
    for n in cuerpo:
        if isinstance(n, (_ast.FunctionDef, _ast.ClassDef)) and n.name == name:
            inicio = min([n.lineno] + [d.lineno for d in n.decorator_list])
            return inicio, n.end_lineno
        if isinstance(n, _ast.Assign) and any(
                isinstance(t, _ast.Name) and t.id == name for t in n.targets):
            return n.lineno, n.end_lineno
        if (isinstance(n, _ast.AnnAssign) and isinstance(n.target, _ast.Name)
                and n.target.id == name):
            return n.lineno, n.end_lineno
    return None

def chequeo_de_deriva(shas=_SHAS, root=ROOT):
    problemas = []
    for key, sha in shas.items():
        rel, name = key.rsplit(':', 1)
        text = (root / rel).read_text(encoding='utf-8')
        lines = text.splitlines(keepends=True)
        sitio = _pieza(_ast.parse(text).body, name)
        if sitio is None:
            problemas.append(f'{key}: ya no existe en musepipe'); continue
        inicio, fin = sitio
        src = ''.join(lines[inicio - 1:fin]).rstrip('\n')
        actual = _hashlib.sha256(src.encode('utf-8')).hexdigest()[:12]
        if actual != sha:
            problemas.append(f'{key}: la copia es {sha}, musepipe tiene {actual}')
    return problemas

_deriva = chequeo_de_deriva()
if _deriva:
    print('DERIVA — la cadena cambió y esta copia se quedó atrás:')
    for p in _deriva:
        print('  ·', p)
    print(f'\nRegenera: python scripts/build_debug_notebooks.py --target {TARGET} A3')
else:
    print(f'sin deriva: las {len(_SHAS)} piezas copiadas son las de musepipe')


## 5 · El espectro ANTES y DESPUÉS de corregir

La figura que dice si la corrección se hizo bien. Para cada reducción que **sí** aplicó corrección se dibuja el espectro de la primaria antes y después, en el mismo eje: arriba el rango completo con las bandas sombreadas, abajo un panel por banda con el **continuo local que la etapa ajusta** (dos medianas laterales de `SIDE_WIDTH_A` Å separadas por un hueco de `GAP_A` Å, interpoladas) y la profundidad medida anotada.

**Cómo se lee:** la corrección está bien cuando el fondo de la banda **sube hasta la línea de continuo** y **nada fuera de las bandas se mueve**. Lo segundo no se deja al ojo: lo mide `verify_outside_bands_unchanged`, la V2 de la propia etapa, que exige ≤ 0.2 % de cambio fuera de las regiones corregidas.

En la reducción cuyo cubo de entrada está borrado el «antes» se **reconstruye** como `después × T` con la curva que la etapa guardó. No es una aproximación: `apply_transmission_to_cube_file` escribe la transmisión ya *forzada* (con las ventanas protegidas a 1), que es exactamente por la que dividió.

Se dibujan también las bandas que A3 **no** mide, del catálogo de `musepipe.telluric_lines` — entre ellas **O₂ A**, la más profunda del rango de MUSE. Que quede fuera de la decisión es el hilo del que tira §7.


In [ ]:
def espectro(cube_path, yx, radius, half=30):
    """(wave, spec) de una apertura, leyendo SOLO una ventana espacial.

    Los cubos pesan 1.4-3.5 GB y aquí solo hacen falta ~60x60 px alrededor de
    la primaria. `_read_primary_spectrum` de la etapa se trae el cubo entero;
    esto da el mismo espectro en ~2 s y sin llenar la memoria.
    """
    cube_path = Path(cube_path)
    with fits.open(cube_path, memmap=True) as h:
        hdu = h['DATA'] if 'DATA' in h else h[1 if h[0].data is None else 0]
        ny, nx = hdu.shape[1:]
        cy, cx = float(yx[0]), float(yx[1])
        y0 = max(0, int(np.floor(cy)) - half); y1 = min(ny, int(np.ceil(cy)) + half + 1)
        x0 = max(0, int(np.floor(cx)) - half); x1 = min(nx, int(np.ceil(cx)) + half + 1)
        win = np.asarray(hdu.data[:, y0:y1, x0:x1], dtype=np.float64)
        wave = wavelength_axis_from_header(hdu.header, hdu.shape[0])
        if not np.all(np.isfinite(wave)) or wave[0] == 0:
            wave = wavelength_axis_from_header(h[0].header, hdu.shape[0])
    # La apertura se recentra a coordenadas de la ventana.
    return wave, extract_aperture_spectrum(win, (cy - y0, cx - x0), radius)


def profundidad_cruda(wave, spec, banda):
    """`measure_telluric_depths` SIN el `max(0.0, depth)` — para ver el signo.

    Los ceros exactos que publica el QC pueden ser negativos recortados, y un
    residuo que cambia de signo no es una banda sin corregir: es una corregida.
    """
    m = window_mask(wave, banda)
    cont = local_continuum_linear(wave, spec, banda,
                                  side_width_A=SIDE_WIDTH_A, gap_A=GAP_A)
    v = m & np.isfinite(spec) & np.isfinite(cont) & (cont != 0)
    if not v.any():
        return float('nan')
    return 100.0 * (1.0 - float(np.nanmedian(spec[v] / cont[v])))


def transmision_en(curva, wave):
    with fits.open(Path(curva)) as h:
        tw = np.asarray(h[1].data['wave_A'], dtype=np.float64)
        tt = np.asarray(h[1].data['transmission'], dtype=np.float64)
    return np.interp(wave, tw, tt)


def par_antes_despues(r):
    """(wave, antes, despues, procedencia) de una reducción que corrigió."""
    if r['yx'] is None or r['radio'] is None:
        return None
    if r['pre'] is not None and r['pre'].exists() and r['post'] is not None and r['post'].exists():
        w, a = espectro(r['pre'], r['yx'], r['radio'])
        _, d = espectro(r['post'], r['yx'], r['radio'])
        return w, a, d, 'los dos cubos, en disco'
    if (r['post'] is not None and r['post'].exists()
            and r['curva'] is not None and r['curva'].exists()):
        w, d = espectro(r['post'], r['yx'], r['radio'])
        return w, d * transmision_en(r['curva'], w), d, 'ANTES reconstruido como después × T'
    return None


from musepipe.telluric_lines import TELLURIC_BANDS as CATALOGO, SEVERITY_ALPHA
COLOR_ESPECIE = {'O2': 'tab:orange', 'H2O': 'tab:cyan'}


def figura_antes_despues(w, antes, despues, titulo, procedencia):
    zooms = list(BANDAS_MAS.items())
    fig = plt.figure(figsize=(12.5, 7.2))
    gs = fig.add_gridspec(2, len(zooms), height_ratios=[1.7, 1.4], hspace=0.42, wspace=0.28)
    ax0 = fig.add_subplot(gs[0, :])
    for b in CATALOGO:
        ax0.axvspan(b['lo_A'], b['hi_A'], color=COLOR_ESPECIE.get(b['species'], 'grey'),
                    alpha=SEVERITY_ALPHA.get(b['severity'], 0.1), zorder=0)
        ax0.annotate(b['name'], ((b['lo_A'] + b['hi_A']) / 2, 0.985),
                     xycoords=('data', 'axes fraction'), ha='center', va='top',
                     fontsize=6, color='0.35')
    for lo, hi in PROTECTED_A:
        ax0.axvspan(lo, hi, color='0.5', alpha=0.16, zorder=0)
    ax0.plot(w, antes, lw=0.6, color='tab:red', label='antes (sin corregir)')
    ax0.plot(w, despues, lw=0.6, color='tab:blue', label='después (corregido)')
    ax0.set_xlim(w[0], w[-1])
    _fin = np.isfinite(antes)
    if _fin.any():
        ax0.set_ylim(np.nanpercentile(antes[_fin], 0.5), np.nanpercentile(antes[_fin], 99.8))
    ax0.set_xlabel('λ [Å]'); ax0.set_ylabel('flujo en la apertura')
    ax0.legend(loc='upper left', fontsize=8, framealpha=0.9)
    ax0.set_title(f'{titulo}\n({procedencia}; gris = ventanas protegidas)', fontsize=9)
    for j, (nombre, banda) in enumerate(zooms):
        ax = fig.add_subplot(gs[1, j])
        lo, hi = banda
        sel = (w >= lo - 2.2 * SIDE_WIDTH_A) & (w <= hi + 2.2 * SIDE_WIDTH_A)
        ax.axvspan(lo, hi, color='tab:orange', alpha=0.15, zorder=0)
        for datos, color, etq in ((antes, 'tab:red', 'antes'), (despues, 'tab:blue', 'después')):
            ax.plot(w[sel], datos[sel], lw=0.7, color=color, label=etq)
            cont = local_continuum_linear(w, datos, banda,
                                          side_width_A=SIDE_WIDTH_A, gap_A=GAP_A)
            ax.plot(w[sel], cont[sel], lw=0.9, ls='--', color=color, alpha=0.65)
        d_a = profundidad_cruda(w, antes, banda); d_d = profundidad_cruda(w, despues, banda)
        ax.set_title(f'{nombre}\nantes {d_a:.2f} % · después {d_d:.2f} %', fontsize=8)
        ax.set_xlim(w[sel][0], w[sel][-1]); ax.tick_params(labelsize=7)
        ax.set_xlabel('λ [Å]', fontsize=7)
        if j == 0:
            ax.legend(fontsize=6.5, loc='lower left')
    # Sin `tight_layout`: el gridspec ya fija los huecos, y mezclarlos avisa.
    plt.show()


PARES = {}
for r in REDUCCIONES:
    if not r['aplicado']:
        continue
    par = par_antes_despues(r)
    if par is None:
        print('sin par antes/después para', r['etiqueta'],
              '— falta el cubo o la curva que declara su QC')
        continue
    w, a, d, proc = par
    PARES[r['etiqueta']] = (w, a, d)
    v2 = verify_outside_bands_unchanged(a, d, w)
    print(f"{r['etiqueta']}: {proc}")
    print(f"   V2 (fuera de banda sin tocar, ≤0.2 %): {'PASA' if bool(v2) else 'FALLA'}")
    figura_antes_despues(w, a, d, f"{TARGET} · {r['etiqueta']}", proc)
if not PARES:
    print('ninguna reducción de este objeto aplicó corrección: no hay antes/después que enseñar')


## 6 · El cubo canónico y su apertura

**Si el QC declara `primary_yx` y `aperture_radius_px`, se usan y punto** — es la procedencia, y desde 2026-07-31 el esquema los escribe. Si no los trae (los QC anteriores a esa fecha, y los dos históricos escritos a mano) se cae al **barrido** que hubo que inventar para recuperarlos: posiciones candidatas (el centro del recorte de B1 redondeado, el centro sin redondear, el pico de luz blanca) × radios, marcando la celda que reproduce el número del QC. La celda dice por cuál de los dos caminos fue: adivinar y saber no son lo mismo, y el QC no debería obligar a lo primero.

Ojo con el marco: la primaria de B3 está en coordenadas del **cubo recortado** (170 px), y el cubo canónico es el de 200 px. El desfase lo declara `stage01_qc.json:crop_bounds_per_cube` y confundirlos mueve la apertura 15 px.

Debajo, las profundidades **sin recortar a cero**. Los `0.0` que publica el QC no son bandas planas: son valores negativos pasados por `max(0.0, depth)`. Un residuo que se va por debajo del continuo tanto como por encima es la firma de una banda **ya corregida**, no de una que nadie tocó.


In [ ]:
CAN = next((r for r in REDUCCIONES if r['canonica']), None)
if CAN is None:
    print('no hay QC de A3 para la cadena activa: nada que reproducir aquí')
else:
    qc01 = json.loads((SD / 'stage01_qc.json').read_text(encoding='utf-8'))
    centro = [float(v) for v in qc01['crop']['center_yx']]
    bounds = (qc01.get('crop_bounds_per_cube') or [{}])[0]
    off = (float(bounds.get('y1', 0)), float(bounds.get('x1', 0)))
    print(f'centro del recorte (marco del cubo entero): {centro}  ·  desfase B1: {off}')

    with fits.open(CAN['pre'], memmap=True) as h:
        _hdu = h['DATA'] if 'DATA' in h else h[1]
        _cy, _cx = int(round(centro[0])), int(round(centro[1]))
        _w = np.asarray(_hdu.data[::37, _cy - 30:_cy + 31, _cx - 30:_cx + 31], dtype=np.float64)
    _med = np.nanmedian(_w, axis=0)
    _pk = np.unravel_index(np.nanargmax(_med), _med.shape)
    pico = (_cy - 30 + int(_pk[0]), _cx - 30 + int(_pk[1]))

    objetivo = CAN['profundidades'].get('O2_B')
    if CAN['yx'] is not None and CAN['radio'] is not None:
        CAN_YX, CAN_R, origen = tuple(CAN['yx']), CAN['radio'], 'DECLARADA en el QC'
    else:
        CAN_YX, CAN_R, origen = (round(centro[0]), round(centro[1])), RADIUS_PX, 'ADIVINADA por barrido'
        print(f'\nel QC no declara la apertura: barrido de O₂ B (%). Objetivo: {objetivo!r}\n')
        radios = [4.0, 6.0, 8.0, 10.0, 12.0]
        print('  posición'.ljust(34) + ''.join(f'r={r:<9.0f}' for r in radios))
        for pos, etq in ((CAN_YX, 'round(centro)'), (tuple(centro), 'centro'), (pico, 'pico luz blanca')):
            fila = ''
            for r_ in radios:
                _, sp = espectro(CAN['pre'], pos, r_)
                v = measure_telluric_depths(_, sp, bands=BANDS_A)['O2_B']
                marca = ' <=' if (objetivo is not None and np.isclose(v, objetivo, rtol=1e-9)) else '   '
                fila += f'{v:8.4f}{marca}'
            print(f'  {etq:16s} {str(tuple(round(float(c), 2) for c in pos)):15s}' + fila)

    WAVE_CAN, SPEC_CAN = espectro(CAN['pre'], CAN_YX, CAN_R)
    print(f'\napertura: yx={CAN_YX} r={CAN_R}   ({origen})')
    if objetivo is not None:
        _repro = measure_telluric_depths(WAVE_CAN, SPEC_CAN, bands=BANDS_A)['O2_B']
        _ok = 'reproduce' if np.isclose(_repro, objetivo, rtol=1e-9) else 'NO reproduce'
        print(f'  O₂ B con esta apertura: {_repro!r}  →  {_ok} el {objetivo!r} del QC')
    print('profundidades SIN recortar a cero:')
    for nombre, banda in BANDAS_MAS.items():
        cruda = profundidad_cruda(WAVE_CAN, SPEC_CAN, banda)
        publicada = CAN['profundidades'].get(nombre)
        nota = '  <- negativa, el QC publica 0.0' if cruda < 0 else ''
        extra = '' if publicada is not None else '  (este QC es anterior a que la etapa la midiera)'
        print(f'  {nombre:9s} {cruda:8.3f} %   publicada: {publicada}{extra}{nota}')


### La misma banda en cada reducción, normalizada a su continuo

Los tres cubos tienen escalas de flujo distintas, así que para verlos juntos se divide cada espectro **por su propio continuo local** — el mismo que ajusta la etapa. En ese eje, `1.0` es «no hay banda» y el fondo del hueco es la transmisión.

Es la lectura directa de si la reducción quedó bien: **si el canónico se parece al «después» de la reducción antigua y no a su «antes», la corrección está hecha** — y entonces su 0.59 % es un residuo, no una banda intacta.


In [ ]:
if CAN is None or not PARES:
    print('hacen falta el cubo canónico y al menos una reducción con antes/después')
else:
    zooms = list(BANDAS_MAS.items())
    fig, axes = plt.subplots(1, len(zooms), figsize=(13.5, 3.4), squeeze=False)
    etq_ref, (w_ref, a_ref, d_ref) = next(iter(PARES.items()))
    for ax, (nombre, banda) in zip(axes[0], zooms):
        lo, hi = banda
        ax.axvspan(lo, hi, color='tab:orange', alpha=0.15, zorder=0)
        ax.axhline(1.0, color='0.55', lw=0.8, ls=':', zorder=1)
        for w_, sp, color, lab in ((w_ref, a_ref, 'tab:red', 'antigua · antes'),
                                   (w_ref, d_ref, 'tab:blue', 'antigua · después'),
                                   (WAVE_CAN, SPEC_CAN, 'k', 'canónica (multi-noche)')):
            cont = local_continuum_linear(w_, sp, banda,
                                          side_width_A=SIDE_WIDTH_A, gap_A=GAP_A)
            sel = (w_ >= lo - 1.2 * SIDE_WIDTH_A) & (w_ <= hi + 1.2 * SIDE_WIDTH_A)
            with np.errstate(invalid='ignore', divide='ignore'):
                razon = np.where(cont != 0, sp / cont, np.nan)
            ax.plot(w_[sel], razon[sel], lw=0.7, color=color, label=lab)
        ax.set_title(nombre, fontsize=9); ax.tick_params(labelsize=7)
        ax.set_xlabel('λ [Å]', fontsize=7); ax.set_ylim(0.45, 1.35)
    axes[0][0].set_ylabel('flujo / continuo local', fontsize=8)
    axes[0][0].legend(fontsize=6.5, loc='lower left')
    fig.suptitle(f'{TARGET} · la misma banda en cada reducción '
                 f'(histórica: {etq_ref})', fontsize=9)
    fig.tight_layout(); plt.show()


## 7 · Hipótesis 1 — el DRS ya las quitó

Si el cubo canónico ya viene con la corrección telúrica aplicada **por exposición**, su 0.59 % no es la profundidad de la banda: es lo que **queda** después de corregir. Y eso se comprueba sin abrir un cubo, mirando qué se le dio a `muse_scipost`.

El SOF es la lista de entradas de cada receta. `STD_TELLURIC` es la tabla de absorción telúrica medida sobre la estrella estándar de esa noche: si está, el DRS divide por ella; si no está, no. La cadena multi-noche la **exige** — `musepipe/reduction/perexp_plan.py` la añade al plan y levanta `PerExposurePlanError` si las cuentas no salen 1/1 por exposición.

El número que decide es **O₂ A**, la más profunda del rango: si las bandas siguieran ahí, ahí se vería. Cuando esta sección se escribió, `decide_telluric` no la miraba; que ahora sí lo haga sale en parte de lo que se midió aquí.


In [ ]:
def censo_sof(carpeta, patron='muse_scipost*.sof'):
    """Cuántos SOF hay y cuántas veces aparece cada etiqueta de calibración."""
    carpeta = Path(carpeta)
    if not carpeta.is_dir():
        return None
    n, tags = 0, {'STD_RESPONSE': 0, 'STD_TELLURIC': 0}
    for sof in sorted(carpeta.glob(patron)):
        n += 1
        etiquetas = [ln.split()[-1] for ln in sof.read_text(encoding='utf-8').splitlines() if ln.strip()]
        for t in tags:
            tags[t] += etiquetas.count(t)
    return dict(n_sof=n, **tags)


print('=== qué se le dio a muse_scipost en cada reducción ===')
for r in REDUCCIONES:
    if r['pre'] is None:
        continue
    # El workdir de la reducción se deduce del cubo que el QC declara:
    #   .../<workdir>/final/DATACUBE_FINAL.fits   (cadena multi-noche)
    #   .../raw_reduction/products/<receta>/DATACUBE_FINAL.fits  (antiguas)
    candidatos = [r['pre'].parent.parent / 'sof',
                  r['pre'].parent.parent.parent / 'sof',
                  r['pre'].parent.parent / 'cubes' / 'sof']
    for c in candidatos:
        censo = censo_sof(c)
        if censo and censo['n_sof']:
            marca = 'CORRIGE tellúrico' if censo['STD_TELLURIC'] else 'NO corrige telúrico'
            print(f"  {r['etiqueta']:44s} {censo['n_sof']:3d} SOF · "
                  f"STD_RESPONSE {censo['STD_RESPONSE']:3d} · STD_TELLURIC {censo['STD_TELLURIC']:3d}  -> {marca}")
            print(f'      {c}')
            break
    else:
        print(f"  {r['etiqueta']:44s} sin SOF en disco (workdir borrado)")

print(f'\n=== O₂ A {tuple(O2_A_BAND)}, la más profunda del rango ===')
print('   (esta sección la midió cuando la etapa no la medía; desde 2026-07-31 A3 sí)')
for etiqueta, (w, a, d) in PARES.items():
    print(f'  {etiqueta:44s} antes {profundidad_cruda(w, a, O2_A_BAND):7.3f} %'
          f'   después {profundidad_cruda(w, d, O2_A_BAND):7.3f} %')
if CAN is not None:
    print(f"  {CAN['etiqueta']:44s} {profundidad_cruda(WAVE_CAN, SPEC_CAN, O2_A_BAND):7.3f} %"
          '   (la cadena canónica no aplicó nada)')


## 8 · Hipótesis 2 — combinar exposiciones a masas de aire distintas

La absorción telúrica crece con la masa de aire: por Beer–Lambert, una banda con transmisión `T₀` a `X₀` tiene `T = T₀^(X/X₀)` a otra `X`. Promediar exposiciones tomadas a masas de aire distintas mezcla profundidades distintas, y podría difuminar la banda.

Se mide, no se supone: las masas de aire salen de las **cabeceras de los originales** (`ESO TEL AIRM START/END`), a través del índice de exposiciones del workdir. Son lecturas de cabecera, no de datos: cuestan décimas de segundo.

La cota se calcula con la banda más profunda medida, llevada exposición a exposición y promediada con el peso real de la combinación (`EXPTIME`). Si el factor que sale es mucho menor que el observado, esta hipótesis no explica nada — y conviene mirar su **signo**.


In [ ]:
def masas_de_aire(workdir):
    """(exposición, noche, X, EXPTIME) leyendo SOLO cabeceras de los originales."""
    idx = Path(workdir) / 'inputs' / 'exposures.json'
    if not idx.exists():
        return []
    data = json.loads(idx.read_text(encoding='utf-8'))
    exps = data['exposures'] if isinstance(data, dict) else data
    filas = []
    for e in exps:
        crudo = (e.get('metadata') or {}).get('raw_object')
        if not crudo or not Path(crudo).exists():
            continue
        h = fits.getheader(crudo, 0)
        x0 = float(h.get('ESO TEL AIRM START', np.nan))
        x1 = float(h.get('ESO TEL AIRM END', np.nan))
        filas.append(dict(exp=e.get('exposure_id', Path(crudo).stem),
                          noche=str(h.get('DATE-OBS', ''))[:10],
                          x=0.5 * (x0 + x1), x0=x0, x1=x1,
                          t=float(h.get('EXPTIME', np.nan))))
    return filas


FILAS = masas_de_aire(CAN['pre'].parent.parent) if CAN is not None else []
if not FILAS:
    print('sin índice de exposiciones en disco: no se puede medir la masa de aire')
else:
    X = np.array([f['x'] for f in FILAS]); T_EXP = np.array([f['t'] for f in FILAS])
    print(f'{len(FILAS)} exposiciones · X de {X.min():.3f} a {X.max():.3f} · '
          f'media {X.mean():.3f} · ponderada por EXPTIME {np.average(X, weights=T_EXP):.3f}')
    for n in sorted({f['noche'] for f in FILAS}):
        sub = [f for f in FILAS if f['noche'] == n]
        print(f"  {n}: {len(sub):2d} exp · EXPTIME {sorted({int(s['t']) for s in sub})} · "
              f"X {min(s['x0'] for s in sub):.3f} -> {max(s['x1'] for s in sub):.3f}")

    fig, ax = plt.subplots(figsize=(9, 3.4))
    for n in sorted({f['noche'] for f in FILAS}):
        idx = [i for i, f in enumerate(FILAS) if f['noche'] == n]
        ax.scatter(idx, [FILAS[i]['x'] for i in idx],
                   s=[max(6.0, FILAS[i]['t'] / 8.0) for i in idx], label=n)
    ax.set_xlabel('exposición (orden del plan de combinación)')
    ax.set_ylabel('masa de aire X')
    ax.set_title('masa de aire por exposición (tamaño = EXPTIME)', fontsize=9)
    ax.legend(fontsize=7); ax.grid(alpha=0.25)
    fig.tight_layout(); plt.show()

    # Cota de Beer-Lambert: la banda más profunda medida, llevada a cada X.
    prof = [(e, profundidad_cruda(w, a, O2_A_BAND) / 100.0) for e, (w, a, _d) in PARES.items()]
    prof = [(e, p) for e, p in prof if np.isfinite(p) and p > 0]
    if not prof:
        print('sin banda de referencia medida: no se puede acotar')
    else:
        etq_ref, d0 = max(prof, key=lambda kv: kv[1])
        ref = next(r for r in REDUCCIONES if r['etiqueta'] == etq_ref)
        x0 = float((ref['qc'].get('fit') or {}).get('airmass_sci') or np.average(X, weights=T_EXP))
        d_comb = 1.0 - np.average((1.0 - d0) ** (X / x0), weights=T_EXP)
        obs = None
        if CAN is not None:
            obs = profundidad_cruda(WAVE_CAN, SPEC_CAN, O2_A_BAND) / 100.0
        print(f'\nreferencia: O₂ A = {100*d0:.2f} % en {etq_ref} (X₀ = {x0:.3f})')
        print(f'combinada sobre las {len(FILAS)} exposiciones reales: {100*d_comb:.2f} %'
              f'  ->  factor {d_comb/d0:.3f}x')
        if obs is not None and obs > 0:
            print(f'factor observado en el cubo canónico: {obs/d0:.4f}x  (1/{d0/obs:.1f})')
            print('la masa de aire ' + ('NO explica la caída: va en el sentido contrario'
                                       if d_comb >= d0 else 'no basta para explicar la caída'))


## 9 · Hipótesis 3 — la mediana sobre toda la banda diluye el núcleo

`measure_telluric_depths` compara la **mediana** de la banda entera con su continuo. Una banda con un núcleo estrecho y profundo entre alas transparentes da una mediana pequeña: el estimador diluye. Es real y explica por qué A3 puede llamar «superficial» a una banda cuyo fondo baja mucho más — pero se cuantifica poniendo la mediana al lado del percentil 10 y del mínimo.


In [ ]:
def dispersion_en_banda(wave, spec, banda):
    m = window_mask(wave, banda)
    cont = local_continuum_linear(wave, spec, banda,
                                  side_width_A=SIDE_WIDTH_A, gap_A=GAP_A)
    v = m & np.isfinite(spec) & np.isfinite(cont) & (cont != 0)
    if not v.any():
        return None
    caida = 100.0 * (1.0 - spec[v] / cont[v])
    return dict(mediana=float(np.nanmedian(caida)), p90=float(np.nanpercentile(caida, 90)),
                maximo=float(np.nanmax(caida)), n=int(v.sum()))


if CAN is not None:
    print(f"{'banda':10s} {'mediana':>9s} {'p90':>9s} {'máximo':>9s}   (% de caída bajo el continuo)")
    print(f"-- cubo canónico ({CAN['etiqueta']})")
    for nombre, banda in BANDAS_MAS.items():
        d = dispersion_en_banda(WAVE_CAN, SPEC_CAN, banda)
        if d:
            print(f"{nombre:10s} {d['mediana']:9.3f} {d['p90']:9.3f} {d['maximo']:9.3f}")
for etiqueta, (w, a, _d) in PARES.items():
    print(f'-- antes de corregir ({etiqueta})')
    for nombre, banda in BANDAS_MAS.items():
        d = dispersion_en_banda(w, a, banda)
        if d:
            print(f"{nombre:10s} {d['mediana']:9.3f} {d['p90']:9.3f} {d['maximo']:9.3f}")
print('\nLa mediana es el estimador de la etapa; el máximo es lo que se ve en la figura de §5.')


## 10 · Hipótesis 4 — el continuo curvado fabrica la profundidad

`local_continuum_linear` no ajusta una recta: toma la **mediana de dos bandas laterales** y las une con `np.interp`. Son dos grados de libertad, así que **no puede representar curvatura**. Si el continuo de la primaria se comba entre los dos laterales, la cuerda pasa por encima del continuo real en el centro de la banda y el estimador anota como absorción algo que es forma del espectro.

La tentación es reajustar con una parábola y quedarse con el número nuevo. **No se hace aquí**, y §11 mide por qué: con solo dos grupos de 40 Å separados por el hueco de la banda, el término cuadrático se extrapola y lo fija el ruido de los laterales.

En su lugar se mide **el sesgo del propio estimador**, sin modelo de continuo: se corre `measure_telluric_depths` **tal cual** sobre trozos del mismo espectro donde el catálogo de `telluric_lines` dice que no hay telúrico, con la **misma** anchura de banda, los **mismos** laterales y el **mismo** hueco. Lo que marque ahí lo ha fabricado de la nada. Es la invariante de la casa: un control procesado igual que el objeto.

**Predicción, escrita antes de mirar** — si la hipótesis es cierta, las ventanas de control alrededor de una banda con veredicto `needed` marcan un sesgo del orden de la profundidad declarada, y al descontarlo la banda baja del umbral. Si marcan cero, la profundidad es real y la hipótesis es **falsa**: eso también se escribe.

El `clip_negative=False` es imprescindible: con el clip puesto la distribución de control se trunca en 0 y su mediana se sesga hacia arriba, o sea que el control subestimaría justo lo que existe para medir.


In [ ]:
# El catalogo de 9 bandas se importa (es un dato, no un calculo): una sola
# definicion de 'donde hay telurico', la misma que usan la figura y G3.
from musepipe.telluric_lines import TELLURIC_BANDS as _H4_CATALOGO
from musepipe.telluric_lines import AO_LASER_WINDOW_A as _H4_LASER

_H4_PASO, _H4_VECINDAD = 10.0, 300.0
#: rasgos estelares fuertes: dentro de una ventana marcarian profundidad
#: que no es curvatura del continuo.
_H4_ESTELARES = ((5880., 5900.), (8490., 8510.), (8530., 8555.), (8650., 8675.))


def _h4_sigma(x):
    """Sigma robusta (MAD). No se importa `robust_sigma`: aqui solo se copian
    literalmente las funciones que usa la cadena, y esta no es una de ellas."""
    x = np.asarray(x, dtype=float); x = x[np.isfinite(x)]
    if x.size < 2:
        return float('nan')
    return float(1.4826 * np.median(np.abs(x - np.median(x))))


def _h4_limpios(wave):
    sucio = np.zeros(np.shape(wave), dtype=bool)
    for _b in _H4_CATALOGO:
        sucio |= window_mask(wave, (_b['lo_A'], _b['hi_A']))
    for _w in tuple(PROTECTED_A) + _H4_ESTELARES + (_H4_LASER,):
        sucio |= window_mask(wave, _w)
    return (~sucio) & np.isfinite(wave)


def _h4_sesgo(wave, spec, banda, limpio):
    """Sesgo local del estimador: mediana de lo que marca en ventanas limpias
    de la MISMA geometria, en el entorno de la banda."""
    lo, hi = banda; ancho = hi - lo; centro = 0.5 * (lo + hi)
    ok = limpio & np.isfinite(spec)
    if not ok.any():
        return None
    lecturas = []
    p = float(wave[ok].min()) + GAP_A + SIDE_WIDTH_A
    tope = float(wave[ok].max()) - ancho - GAP_A - SIDE_WIDTH_A
    while p <= tope:
        tramo = window_mask(wave, (p - GAP_A - SIDE_WIDTH_A,
                                   p + ancho + GAP_A + SIDE_WIDTH_A))
        if tramo.any() and ok[tramo].all():
            d = measure_telluric_depths(wave, spec, bands={'c': (p, p + ancho)},
                                        side_width_A=SIDE_WIDTH_A, gap_A=GAP_A,
                                        clip_negative=False)['c']
            if np.isfinite(d):
                lecturas.append((p + ancho / 2.0, d))
        p += _H4_PASO
    if len(lecturas) < 3:
        return None
    cen = np.array([c for c, _ in lecturas]); val = np.array([d for _, d in lecturas])
    cerca = np.abs(cen - centro) <= _H4_VECINDAD
    if cerca.sum() < 3:
        cerca = np.abs(cen - centro) <= 2 * _H4_VECINDAD
    if cerca.sum() < 3:
        return None
    azul, rojo = cerca & (cen < centro), cerca & (cen > centro)
    # dos terminos empiricos: dispersion entre ventanas, y la deriva azul-rojo
    # (la curvatura cambia con lambda y la banda cae entre los dos vecindarios)
    d_lambda = (0.5 * abs(np.median(val[azul]) - np.median(val[rojo]))
                if azul.any() and rojo.any() else 0.0)
    return dict(sesgo=float(np.median(val[cerca])),
                sigma=float(np.hypot(_h4_sigma(val[cerca]), d_lambda)),
                n=int(cerca.sum()), azul=int(azul.sum()), rojo=int(rojo.sum()))


if CAN is not None and 'WAVE_CAN' in globals():
    _h4_lim = _h4_limpios(WAVE_CAN)
    _h4_med = measure_telluric_depths(WAVE_CAN, SPEC_CAN, bands=BANDS_A,
                                      clip_negative=False)
    print(f"{int(_h4_lim.sum())} canales limpios de {WAVE_CAN.size}\n")
    print(f"{'banda':10s} {'medida':>8s} {'sesgo':>8s} {'sigma':>7s} "
          f"{'descontada':>10s}   ventanas (azules/rojas)")
    _h4_out = {}
    for _n, _b in BANDS_A.items():
        _s = _h4_sesgo(WAVE_CAN, SPEC_CAN, _b, _h4_lim)
        _h4_out[_n] = _s
        if _s is None:
            print(f'{_n:10s} {_h4_med[_n]:8.3f}   sin ventanas de control limpias')
            continue
        print(f"{_n:10s} {_h4_med[_n]:8.3f} {_s['sesgo']:+8.3f} {_s['sigma']:7.3f} "
              f"{_h4_med[_n] - _s['sesgo']:+10.3f}   "
              f"{_s['n']} ({_s['azul']}/{_s['rojo']})")
    print('\nOJO: una profundidad NEGATIVA es una banda ya corregida (o corregida de\n'
          'mas). El QC de la etapa la guarda como 0.0 por el clip, asi que ese 0.0 es\n'
          'un valor CENSURADO, no una medida: no se puede leer como residuo nulo.')


### 10.1 · Control positivo — ¿es capaz esta prueba de ver un sesgo?

Una prueba que devuelve cero no vale nada hasta que se demuestra que sabe devolver otra cosa. Se inyecta una **curvatura suave sobre todo el rango** —que es lo que físicamente es «una primaria más roja»— y se comprueba que las ventanas de control la recuperan. La comba se aplica al espectro entero, no solo a la banda: una comba centrada en la banda no la verían los vecinos, y el control saldría ciego **sin que eso signifique que no hay sesgo**.


In [ ]:
if CAN is not None and 'WAVE_CAN' in globals():
    _h4_banda = BANDS_A['O2_A'] if 'O2_A' in BANDS_A else list(BANDS_A.values())[0]
    _h4_c0 = 0.5 * (_h4_banda[0] + _h4_banda[1])
    _h4_base = _h4_sesgo(WAVE_CAN, SPEC_CAN, _h4_banda, _h4_lim)
    print(f"{'curvatura':>10s} {'inyectado':>10s} {'recuperado':>11s} {'error':>8s}   (%)")
    for _c in (0.0, 0.02, 0.06, 0.12):
        _comba = 1.0 + _c * ((WAVE_CAN - _h4_c0) / 1000.0) ** 2
        _iny = measure_telluric_depths(WAVE_CAN, np.ones_like(WAVE_CAN) * _comba,
                                       bands={'b': _h4_banda}, clip_negative=False)['b']
        _s = _h4_sesgo(WAVE_CAN, SPEC_CAN * _comba, _h4_banda, _h4_lim)
        if _s is None or _h4_base is None:
            print('    sin ventanas: el control positivo no se puede correr'); break
        _rec = _s['sesgo'] - _h4_base['sesgo']
        print(f'{_c:10.3f} {_iny:10.3f} {_rec:11.3f} {_rec - _iny:+8.3f}')
    print('\nSi la columna «recuperado» sigue a «inyectado», la prueba tiene\n'
          'sensibilidad y un cero suyo es un cero de verdad. Fijate tambien en\n'
          'CUANTO fabrica una curvatura global: es el orden de magnitud que una\n'
          'primaria mas roja puede explicar, y acota la hipotesis entera.')


## 11 · De dónde sale el número de la parábola

El ajuste polinómico solo dispone de **dos grupos de 40 Å** separados por el hueco de la banda. Una recta está bien determinada por esa geometría; una parábola tiene que **extrapolar** su curvatura a través del hueco, y ahí quien manda es el ruido de los laterales. Se cuantifica remuestreando los laterales con su propio ruido empírico y mirando cuánto baila cada grado.

Si la barra de la parábola es comparable a la diferencia entre la parábola y la cuerda, entonces «con parábola sale menos» **no es una medida**, es la dispersión del estimador.


In [ ]:
_H5_SEMILLA = 20260802   # determinista, como todo lo que decide en esta cadena


def _h5_depth_poly(wave, spec, banda, grado):
    lo, hi = banda
    lados = (((wave >= lo - GAP_A - SIDE_WIDTH_A) & (wave <= lo - GAP_A)) |
             ((wave >= hi + GAP_A) & (wave <= hi + GAP_A + SIDE_WIDTH_A))) & np.isfinite(spec)
    if lados.sum() < grado + 2:
        return float('nan'), lados
    p = np.polynomial.Polynomial.fit(wave[lados], spec[lados], grado)
    return measure_telluric_depths(wave, spec, bands={'b': banda},
                                   continuum=lambda w, s, b: p(w),
                                   clip_negative=False)['b'], lados


if CAN is not None and 'WAVE_CAN' in globals():
    _rng = np.random.default_rng(_H5_SEMILLA)
    for _n, _b in BANDS_A.items():
        _, _lados = _h5_depth_poly(WAVE_CAN, SPEC_CAN, _b, 1)
        if _lados.sum() < 6:
            print(f'{_n}: laterales insuficientes'); continue
        _rec = np.polynomial.Polynomial.fit(WAVE_CAN[_lados], SPEC_CAN[_lados], 1)
        _sig = _h4_sigma(SPEC_CAN[_lados] - _rec(WAVE_CAN[_lados]))
        _niv = float(np.median(SPEC_CAN[_lados]))
        print(f'-- {_n}  (ruido lateral {100*_sig/_niv:.2f} % del nivel) --')
        for _g, _et in ((1, 'recta   '), (2, 'parabola'), (3, 'cubica  ')):
            _cen, _ = _h5_depth_poly(WAVE_CAN, SPEC_CAN, _b, _g)
            _m = []
            for _ in range(200):
                _r = SPEC_CAN.copy()
                _r[_lados] = SPEC_CAN[_lados] + _rng.normal(0., _sig, int(_lados.sum()))
                _m.append(_h5_depth_poly(WAVE_CAN, _r, _b, _g)[0])
            print(f'   {_et} {_cen:+8.3f}  ±{np.nanstd(_m):6.3f}')
        _cu = measure_telluric_depths(WAVE_CAN, SPEC_CAN, bands={'b': _b},
                                     clip_negative=False)['b']
        _m = []
        for _ in range(200):
            _r = SPEC_CAN.copy()
            _r[_lados] = SPEC_CAN[_lados] + _rng.normal(0., _sig, int(_lados.sum()))
            _m.append(measure_telluric_depths(WAVE_CAN, _r, bands={'b': _b},
                                              clip_negative=False)['b'])
        print(f'   cuerda   {_cu:+8.3f}  ±{np.nanstd(_m):6.3f}   <- el estimador de la etapa')


## 12 · Comparación con la cadena

Tres reproducciones, una por reducción, cada una contra **su** QC:

1. **canónica** — `measure_telluric_depths` sobre el cubo multi-noche con las perillas resueltas y la apertura recuperada en §6, contra `depth_pct_by_band` a `rtol=1e-9`; y `decide_telluric` contra el veredicto;
2. **con los dos cubos en disco** — antes y después, contra las profundidades declaradas y contra la V1 del QC, a la precisión con que el QC las guardó;
3. **con el cubo de entrada borrado** — el «antes» reconstruido como `después × T`.

`IDÉNTICO` solo sale si pasan **todas** las comparaciones disponibles. Si falta un cubo se dice cuál y no se imprime: un hueco no es un acuerdo.


In [ ]:
def _tolerancia(valor):
    """El QC guarda algunos números redondeados: se compara a SU precisión."""
    txt = repr(float(valor))
    dec = len(txt.split('.')[1]) if '.' in txt else 0
    return 0.5 * 10.0 ** (-dec) if dec <= 6 else 0.0


def compara(nombre, mio, suyo, rtol=1e-9):
    if suyo is None:
        print(f'    {nombre:26s} el QC no lo declara'); return True
    tol = _tolerancia(suyo)
    ok = bool(np.isclose(float(mio), float(suyo), rtol=rtol, atol=tol))
    print(f'    {nombre:26s} {float(mio):12.6f}  QC {float(suyo):12.6f}'
          f"{'' if ok else '   <-- DIFIERE'}")
    return ok


ok = True; comparadas = 0
for r in REDUCCIONES:
    print(('· CANÓNICA  ' if r['canonica'] else '· histórica ') + r['etiqueta'])
    if r['canonica']:
        if CAN is None or 'WAVE_CAN' not in globals():
            print('    sin cubo: no se compara'); ok = False; continue
        mias = measure_telluric_depths(WAVE_CAN, SPEC_CAN, bands=BANDS_A)
        for banda, suyo in r['profundidades'].items():
            ok &= compara(banda, mias.get(banda, float('nan')), suyo); comparadas += 1
        dec = decide_telluric(mias, science_needs_red_continuum=NEEDS_RED_CONT,
                              threshold_pct=THRESHOLD_PCT)
        suyo = r['qc'].get('decision', {})
        for campo, mio, ref in (('veredicto', dec.decision, suyo.get('verdict')),
                                ('aplicado', dec.telluric_applied, suyo.get('telluric_applied')),
                                ('checkpoint', dec.checkpoint_required, suyo.get('checkpoint_required'))):
            igual = (ref is None) or (mio == ref)
            print(f"    {campo:26s} {str(mio):>12s}  QC {str(ref):>12s}"
                  f"{'' if igual else '   <-- DIFIERE'}")
            ok &= igual; comparadas += 1
        continue
    par = PARES.get(r['etiqueta'])
    if par is None:
        print('    sin par antes/después en disco: no se compara'); ok = False; continue
    w, antes, despues = par
    mias = measure_telluric_depths(w, antes, bands=BANDS_A)
    for banda, suyo in r['profundidades'].items():
        ok &= compara('antes · ' + banda, mias.get(banda, float('nan')), suyo); comparadas += 1
    v1 = (r['qc'].get('verification') or {}).get('v1_o2_depth_pre_post_pct')
    if isinstance(v1, (list, tuple)) and len(v1) == 2:
        post = measure_telluric_depths(w, despues, bands=BANDS_A)
        ok &= compara('después · O2_B (V1)', post.get('O2_B', float('nan')), v1[1]); comparadas += 1

print()
if ok and comparadas:
    print('IDÉNTICO: la copia reproduce la cadena.')
else:
    print('DIFIERE — si has tocado una perilla, es lo esperado; '
          'si no, revisa el chequeo de deriva y qué cubos faltan.')


## 13 · Conclusión

Las cuatro hipótesis sobre por qué una banda telúrica mide lo que mide, y lo que dice de cada una **el objeto y la reducción que este notebook tenga delante** — los números concretos los imprimen las celdas, no este texto:

1. **El DRS ya las quitó** (§7). Cuando la cadena le pasa `STD_TELLURIC` a `muse_scipost` en cada exposición, la banda llega al cubo combinado ya corregida y lo que A3 mide es un **residuo**, no la absorción. Es lo que separa una reducción multi-noche de las antiguas de una sola noche, y se ve comparando las reducciones de §2 cuando hay más de una.
2. **La masa de aire contribuye, pero no explica** (§8). Beer–Lambert sobre las exposiciones reales, pesadas por `EXPTIME`, da un factor de orden 1; combinar a masas de aire mayores dejaría la banda *algo más profunda*, no veinte veces menos.
3. **La mediana diluye el núcleo** (§9). El estimador compara la mediana de la banda entera con su continuo, no el fondo: una banda con núcleo estrecho y alas transparentes da un número pequeño aunque su mínimo baje mucho más.
4. **El continuo curvado fabrica profundidad** (§10). La cuerda entre las dos medianas laterales no puede seguir una curvatura, y lo que se sale se anota como absorción. **Cuánto**, en esta banda y en este objeto, lo mide la tabla de ventanas de control — con su control positivo al lado, que es lo que permite leer un cero suyo como un cero de verdad y no como ceguera del método.

### Cómo se lee la tabla de §10

- **sesgo ≈ profundidad medida** → la banda es sobre todo forma del continuo, y descontarlo la baja del umbral. La decisión de A3 estaría apoyada en un artefacto.
- **sesgo ≈ 0 con el control positivo respondiendo** → la profundidad es real. Que *una parábola* dé menos (§11) no la contradice: hay que mirar antes la barra de la parábola, que sale del ruido de los laterales extrapolado a través del hueco.
- **σ del sesgo comparable al propio sesgo** → el estimador no resuelve el umbral en este objeto, y eso es en sí mismo el resultado: lo que procede es declarar el residuo como sistemático, no afinar el estimador hasta que dé el número deseado.

### Lo que este notebook deja medido, y que el QC no guarda

- **El QC redondea a cero las profundidades negativas** (`max(0.0, ·)`). Una banda *ya corregida* —o corregida de más— mide **negativo**, y esa es justamente la prueba de que el DRS actuó. Aquí se lee con `clip_negative=False`: donde el QC pone `0.0` puede haber un número negativo con información, así que ese `0.0` es un valor **censurado** y no puede citarse como «residuo nulo» en un presupuesto de error.
- **El sesgo del estimador no viaja en el QC.** La decisión de A3 se toma sobre `depth_pct_by_band` sin ninguna estimación de cuánto de eso fabrica el método; desde el QC solo no se puede saber.

### Un defecto de procedencia, si aparece arriba

Si §6 ha tenido que **recuperar la apertura por barrido**, es que ese QC no declara `primary_yx`/`aperture_radius_px` y sus números no son reproducibles a partir del QC solo. Eso es arqueología, no procedencia; el esquema nuevo ya los declara.


## 14 · molecfit: las dos granularidades, medidas

Hasta aquí el notebook explica **qué mide** el estimador de A3. Esta sección responde a otra pregunta, la que abre [`docs/2026-08-06_indicaciones_reduccion.md`](../../../docs/2026-08-06_indicaciones_reduccion.md) §5: si molecfit se ajusta **por exposición** o **sobre el cubo ya combinado**. La indicación es no elegir a priori — se hacen las dos y gana la que mida mejor.

Va aquí y no en un notebook aparte porque usa las mismas funciones numéricas ya copiadas arriba (`local_continuum_linear`, `window_mask`), y así queda bajo el mismo chequeo de deriva de §4. Y es barata: lee espectros de 3681 canales, no los cubos.

**Los productos no los genera este notebook**, porque son ~10 min de `esorex`. Los genera `scripts/molecfit_granularity.py` y aquí solo se leen; si no están, la celda lo dice e imprime la orden. La raíz sale de `MUSE_MOLECFIT_ROOT` si está en el entorno, y si no de `runs/<RUN>/molecfit_granularity`.

> **Por qué molecfit y no seguir con `STD_TELLURIC`:** en O₂ A el residuo por canal de la corrección actual es ~4× el suelo de ruido y **no se repite entre reducciones**, o sea que es artefacto de dividir por otro espectro observado y no física. O₂ A es un bosque de líneas **saturadas** que MUSE no resuelve, y contra eso el escalado en masa de aire `T^(X/X_std)` solo vale para líneas finas y cualquier desajuste de sub-píxel en λ produce el patrón alterno sobre/infra-corregido. molecfit ajusta la atmósfera línea a línea y ajusta λ y la LSF.


In [ ]:
import os

MOLECFIT_ROOT = Path(os.environ.get('MUSE_MOLECFIT_ROOT') or (RD / 'molecfit_granularity'))
_manifiesto = MOLECFIT_ROOT / 'manifest.json'
MF = json.loads(_manifiesto.read_text(encoding='utf-8')) if _manifiesto.exists() else None

if MF is None:
    print('no hay ajustes de molecfit en', MOLECFIT_ROOT)
    print('se generan (~10 min) con:\n')
    print('  python scripts/molecfit_granularity.py \\\n'
          '      --out <dir> --combined <cubo_combinado> \\\n'
          '      --exposures <cubo_exp1> ... \\\n'
          '      --prep-dir <raw_reduction/molecfit_a1a> --raw-dir <crudos>\n')
    print('y se leen apuntando MUSE_MOLECFIT_ROOT a <dir>.')
else:
    print(f'{len(MF)} ajustes en {MOLECFIT_ROOT}\n')
    cab = ('etiqueta', 'DATE-OBS', 't[s]', 'X_mid', 'alt', 'RH%', 'st', 'iter',
           'chi2_ini', 'chi2_best', 'rel_O2', 'H2O[mm]')
    print('{:<10s}{:<21s}{:>7s}{:>8s}{:>7s}{:>6s}{:>4s}{:>6s}{:>11s}{:>11s}{:>8s}{:>9s}'.format(*cab))
    for f in MF:
        print('{:<10s}{:<21s}{:>7.0f}{:>8s}{:>7.2f}{:>6.1f}{:>4s}{:>6s}{:>11s}{:>11s}{:>8s}{:>9s}'.format(
            f['etiqueta'], str(f['date_obs'])[:19], f['exptime'] or 0,
            'n/d' if f['X_mid'] is None else f"{f['X_mid']:.3f}", f['altitud_deg'],
            f['humedad_pct'],
            'n/d' if f.get('status') is None else f"{f['status']:.0f}",
            'n/d' if f.get('iterations') is None else f"{f['iterations']:.0f}",
            'n/d' if f.get('initial_chi2') is None else f"{f['initial_chi2']:.1f}",
            'n/d' if f.get('best_chi2') is None else f"{f['best_chi2']:.1f}",
            'n/d' if f.get('rel_mol_col_O2') is None else f"{f['rel_mol_col_O2']:.4f}",
            'n/d' if f.get('h2o_col_mm') is None else f"{f['h2o_col_mm']:.3f}"))
        if f.get('aviso'):
            print(f"           AVISO: {f['aviso']}")
    # `status` 1-3 = convergencia; 4 con chi2_ini == chi2_best es el ajuste
    # CONGELADO que produce el `WLC_CONST` por defecto (-0.05). Se mira, no
    # se supone: un ajuste que no se movio no es una medida.
    _quietos = [f['etiqueta'] for f in MF
                if f.get('best_chi2') is not None
                and abs(f['best_chi2'] - f['initial_chi2']) < 1e-6]
    print('\najustes que NO se movieron:', _quietos or 'ninguno')


### 14.1 · Las curvas, una por exposición y la del combinado

`molecfit_model` deja la transmisión **solo dentro de las ventanas que ajustó** (`mrange > 0`): O₂ B y O₂ A. La de rango completo la daría `molecfit_calctrans`, que en esta build está bloqueada — así que lo que se compara es la banda, que es donde está el problema.

**Qué mirar:** el abanico entre exposiciones *es* la variación real de masa de aire dentro del OB. Si la curva del combinado cae fuera de ese abanico, o pegada a un extremo, es que representa a una exposición y no al conjunto.


In [ ]:
if MF is None:
    print('sin ajustes: nada que dibujar')
else:
    _c = np.load(MOLECFIT_ROOT / 'curvas.npz')
    W = _c['wave_A']
    exps = [f['etiqueta'] for f in MF if f['etiqueta'] != 'combinado']
    X = {f['etiqueta']: f['X_mid'] for f in MF}
    bandas = [(k, v) for k, v in BANDAS_MAS.items() if k in ('O2_B', 'O2_A')]
    fig, axes = plt.subplots(1, len(bandas), figsize=(12.5, 3.8), squeeze=False)
    finitos = [x for x in (X[e] for e in exps) if x]
    norma = plt.Normalize(min(finitos), max(finitos)) if finitos else None
    for ax, (nombre, banda) in zip(axes[0], bandas):
        lo, hi = banda
        for e in exps:
            t, r = _c.get(f'mtrans_{e}'), _c.get(f'mrange_{e}')
            if t is None:
                continue
            m = (r > 0) & (W >= lo) & (W <= hi)
            color = plt.cm.viridis(norma(X[e])) if (norma and X[e]) else '0.6'
            ax.plot(W[m], t[m], lw=0.7, color=color, alpha=0.9)
        t, r = _c.get('mtrans_combinado'), _c.get('mrange_combinado')
        if t is not None:
            m = (r > 0) & (W >= lo) & (W <= hi)
            ax.plot(W[m], t[m], lw=1.6, color='crimson', label='combinado')
        ax.set_title(nombre, fontsize=9); ax.set_xlabel('λ [Å]', fontsize=8)
        ax.tick_params(labelsize=7); ax.set_ylim(0, 1.05)
    axes[0][0].set_ylabel('transmisión molecfit', fontsize=8)
    axes[0][0].legend(fontsize=7, loc='lower left')
    if norma is not None:
        fig.colorbar(plt.cm.ScalarMappable(norm=norma, cmap='viridis'),
                     ax=axes[0], label='masa de aire (mitad de exposición)')
    plt.show()

    # El abanico, en numeros: transmision MEDIA en la banda, por ajuste.
    for nombre, banda in bandas:
        lo, hi = banda
        vals = {}
        for e in exps + ['combinado']:
            t, r = _c.get(f'mtrans_{e}'), _c.get(f'mrange_{e}')
            if t is None:
                continue
            m = (r > 0) & (W >= lo) & (W <= hi)
            if m.any():
                vals[e] = float(np.mean(t[m]))
        porexp = [v for k, v in vals.items() if k != 'combinado']
        if not porexp:
            continue
        dentro = (min(porexp) <= vals.get('combinado', np.nan) <= max(porexp))
        print(f'{nombre}: T media por exposición {min(porexp):.4f}–{max(porexp):.4f} '
              f'· combinado {vals.get("combinado", float("nan")):.4f} '
              f'· {"DENTRO" if dentro else "FUERA"} del abanico')


### 14.2 · El dato contra el modelo, observación por observación

Antes de mirar nada corregido: **el espectro sin corregir con el modelo telúrico encima**, un panel por observación para que no se solapen ocho curvas. Es la comprobación que se le hace siempre a un ajuste de molecfit — si el modelo no sigue al dato, lo que venga después no vale.

Los dos van normalizados por **el mismo continuo, el que ajustó el propio molecfit** (`mscal` en `MOLECFIT_DATA`), no por la cuerda de A3: así el dato y el modelo son directamente comparables y lo que se ve entre ellos es error de modelo, no diferencia entre dos estimadores de continuo.

- **negro** — `flujo / mscal`, el dato sin corregir.
- **naranja** — `mtrans`, la transmisión modelada: **las líneas que molecfit dice que hay**.

Solo se dibuja dentro de los rangos ajustados (`mrange > 0`), que es donde el modelo existe. El `rms` anotado en cada panel es el del dato menos el modelo: es la cifra que dice si ese ajuste, en esa banda, describe la observación.

**Qué mirar, por orden.** Primero si el modelo sigue la **envolvente** de la banda —el núcleo profundo y la subida— que es lo que fija la columna de O₂. Después, y es lo que se escapa a simple vista, si reproduce la **amplitud de los dientes**: son líneas que MUSE **no resuelve**, así que su profundidad aparente la fija el núcleo instrumental (la LSF), no la atmósfera. Un modelo que acierte la envolvente y se quede corto en los dientes deja un residuo que **no depende de la masa de aire** y que por tanto no se cierra al combinar — que es exactamente el que aparece en §14.3.

Por eso debajo de las figuras va el **núcleo que ajustó cada uno**, con la LSF real de MUSE al lado como referencia. Un `gauss` pegado a **0** no es una medida: es un parámetro contra su tope, absorbiendo lo que no se está modelando.


In [ ]:
CURVAS = np.load(MOLECFIT_ROOT / 'curvas.npz') if MF is not None else None

def _espectro_de(etq):
    """El espectro de apertura tal cual se le paso a molecfit."""
    with fits.open(MOLECFIT_ROOT / etq / 'science.fits') as h:
        d = h[1].data
        return (np.asarray(d['WAVE'], dtype=np.float64),
                np.asarray(d['FLUX'], dtype=np.float64))

def _datos_molecfit(etq):
    """(lambda EN AIRE, dato normalizado, modelo, rango) del producto.

    `mscal` ES el continuo que ajusto molecfit, asi que `flux/mscal` y
    `mtrans` viven en el mismo eje y su diferencia es error de modelo.

    El eje NO se toma de `MOLECFIT_DATA.lambda`: esa columna sale en VACIO
    —comprobado contra Edlen(1966) sobre el eje de entrada, coincide a
    0.00000 A— y a 7600 A eso son 2.09 A = 1.67 canales al rojo. Las bandas
    de A3 estan en AIRE, asi que usarla correria el sombreado y las
    mascaras. Las filas de salida van 1:1 con las de entrada, luego el eje
    del propio espectro es exacto.
    """
    lam, _ = _espectro_de(etq)
    with fits.open(MOLECFIT_ROOT / etq / 'out' / 'MOLECFIT_DATA.fits') as h:
        d = h[1].data
        flujo = np.asarray(d['flux'], dtype=np.float64)
        escala = np.asarray(d['mscal'], dtype=np.float64)
        modelo = np.asarray(d['mtrans'], dtype=np.float64)
        rango = np.asarray(d['mrange'], dtype=np.float64)
    with np.errstate(invalid='ignore', divide='ignore'):
        dato = np.where(escala != 0, flujo / escala, np.nan)
    return lam, dato, modelo, rango

if MF is None:
    print('sin ajustes: nada que dibujar')
else:
    orden = [f['etiqueta'] for f in MF if f['etiqueta'] != 'combinado'] + ['combinado']
    X = {f['etiqueta']: f['X_mid'] for f in MF}
    bandas = [(k, v) for k, v in BANDAS_MAS.items() if k in ('O2_B', 'O2_A')]
    for nombre, banda in bandas:
        lo, hi = banda
        cols = 4
        filas = int(np.ceil(len(orden) / cols))
        fig, axes = plt.subplots(filas, cols, figsize=(13.5, 2.5 * filas),
                                 squeeze=False, sharex=True, sharey=True)
        for k, etq in enumerate(orden):
            ax = axes[k // cols][k % cols]
            lam, dato, modelo, rango = _datos_molecfit(etq)
            m = (rango > 0) & (lam >= lo) & (lam <= hi)
            if not m.any():
                ax.set_axis_off()
                continue
            ax.plot(lam[m], dato[m], lw=0.8, color='k', label='dato / mscal')
            ax.plot(lam[m], modelo[m], lw=0.9, color='tab:orange',
                    alpha=0.85, label='modelo (mtrans)')
            rms = float(np.nanstd(dato[m] - modelo[m]))
            equis = '' if X[etq] is None else f'X={X[etq]:.3f}  '
            ax.set_title(f'{etq}   {equis}rms={rms:.4f}', fontsize=8,
                         color='crimson' if etq == 'combinado' else 'black')
            ax.tick_params(labelsize=7)
            if k // cols == filas - 1:
                ax.set_xlabel('λ [Å]', fontsize=8)
            if k % cols == 0:
                ax.set_ylabel('transmisión', fontsize=8)
        for k in range(len(orden), filas * cols):
            axes[k // cols][k % cols].set_axis_off()
        axes[0][0].legend(fontsize=6.5, loc='lower left')
        fig.suptitle(f'{TARGET} · {nombre} — dato sin corregir contra el modelo '
                     f'telúrico, por observación', fontsize=9)
        fig.tight_layout()
        plt.show()

    # El nucleo (LSF) que ajusto cada uno. Va aqui porque es lo que decide si
    # el modelo puede reproducir la PROFUNDIDAD de unas lineas que MUSE no
    # resuelve: si el nucleo no es el del instrumento, el modelo sale con la
    # forma equivocada por mucho que acierte la columna de O2.
    print(f'{"ajuste":<11s}{"box[px]":>9s}{"gauss[px]":>11s}{"lorentz[px]":>13s}'
          f'{"chi2_red":>10s}')
    for etq in orden:
        t = fits.getdata(MOLECFIT_ROOT / etq / 'out' / 'BEST_FIT_PARAMETERS.fits', 1)
        nom, val = t.columns.names[:2]
        o = {str(r[nom]).strip(): float(r[val]) for r in t}
        print(f"{etq:<11s}{o.get('boxfwhm', float('nan')):>9.4f}"
              f"{o.get('gaussfwhm', float('nan')):>11.4f}"
              f"{o.get('lorentzfwhm', float('nan')):>13.4f}"
              f"{o.get('reduced_chi2', float('nan')):>10.2f}")
    # La LSF de MUSE a 7600 A ronda 2.6 A = ~2.1 px de 1.25 A. Un nucleo muy
    # por debajo de eso, o un `gauss` pegado a 0, es un parametro en su tope:
    # no esta medido, esta absorbiendo.
    print('\nreferencia: la LSF de MUSE a 7600 Å ≈ 2.6 Å ≈ 2.1 px')


### 14.3 · Cada observación corregida, y el combinado

El equivalente de la figura de §6 —«la misma banda en cada reducción, normalizada a su continuo»— pero para las **exposiciones corregidas por molecfit**: cada una dividida por **su propia** transmisión, y después por su propio continuo local. En ese eje `1.0` es «no queda banda».

**Cómo se lee.** Antes de corregir, las siete curvas se abren en abanico: la banda es más profunda cuanto mayor la masa de aire, y ese abanico *es* la señal telúrica. Si molecfit hace su trabajo, después de corregir el abanico **se cierra** y las siete se superponen.

Y ojo con la lectura fácil de lo que sobra: **una estructura que quede y sea común a las siete no es, por sí sola, prueba de que sea de la estrella.** Es común todo lo que no depende de la masa de aire — y eso incluye un error de modelo que se repite igual en los siete ajustes, porque comparten LSF, ventanas y el mismo tratamiento de un bosque de líneas que MUSE no resuelve. Lo único que el cierre del abanico demuestra es que **la parte que escalaba con la masa de aire ya no está**. Separar «rasgo fotosférico» de «límite del modelo» necesita otra cosa: una plantilla del tipo espectral, o el mismo ejercicio sobre una estrella distinta.

Debajo, ese cierre en números: la **dispersión entre exposiciones canal a canal** dentro de la banda, antes y después. Es una medida distinta de la de §14.4 — allí cada modo se compara contra su propio continuo, aquí las exposiciones se comparan **entre ellas**.

> El continuo local se ajusta sobre bandas laterales que caen **fuera** del rango que molecfit ajustó, así que el cociente es núcleo corregido sobre laterales sin tocar — que es exactamente lo que se quiere medir. El eje x llega a ±60 Å de la banda para que **se vean los laterales que definen el 1.0**.


In [ ]:
def _corregido(etq):
    """Ese espectro dividido por SU transmision, donde molecfit la ajusto.

    Fuera de los rangos ajustados (`mrange == 0`) no hay transmision que
    aplicar y el espectro se deja intacto: `molecfit_calctrans`, que daria
    la curva de rango completo, esta bloqueada en esta build.
    """
    w, s = _espectro_de(etq)
    t, r = CURVAS.get(f'mtrans_{etq}'), CURVAS.get(f'mrange_{etq}')
    fuera = s.copy()
    if t is not None:
        m = (r > 0) & (t > 0)
        fuera[m] = s[m] / t[m]
    return w, fuera

def _razon(w, s, banda):
    """Espectro / su continuo local: el eje donde 1.0 es «no hay banda»."""
    cont = local_continuum_linear(w, s, banda,
                                  side_width_A=SIDE_WIDTH_A, gap_A=GAP_A)
    with np.errstate(invalid='ignore', divide='ignore'):
        return np.where(cont != 0, s / cont, np.nan)

if MF is None:
    print('sin ajustes: nada que dibujar')
else:
    exps = [f['etiqueta'] for f in MF if f['etiqueta'] != 'combinado']
    X = {f['etiqueta']: f['X_mid'] for f in MF}
    bandas = [(k, v) for k, v in BANDAS_MAS.items() if k in ('O2_B', 'O2_A')]
    _x = [v for v in (X[e] for e in exps) if v]
    norma = plt.Normalize(min(_x), max(_x)) if _x else None

    fig, axes = plt.subplots(2, len(bandas), figsize=(12.5, 6.6), squeeze=False,
                             sharex='col')
    for j, (nombre, banda) in enumerate(bandas):
        lo, hi = banda
        for fila, corregir in ((0, False), (1, True)):
            ax = axes[fila][j]
            ax.axvspan(lo, hi, color='tab:orange', alpha=0.13, zorder=0)
            ax.axhline(1.0, color='0.55', lw=0.8, ls=':', zorder=1)
            vistos = []
            for e in exps:
                w, s = _corregido(e) if corregir else _espectro_de(e)
                # +-60 A: los laterales que definen el 1.0 estan a (gap, gap+side)
                # = (10, 50) A de la banda, y con menos margen no se ven.
                sel = (w >= lo - 60) & (w <= hi + 60)
                r = _razon(w, s, banda)[sel]
                color = plt.cm.viridis(norma(X[e])) if (norma and X[e]) else '0.6'
                ax.plot(w[sel], r, lw=0.7, color=color, alpha=0.9)
                vistos.append(r)
            w_c, s_c = _corregido('combinado') if corregir else _espectro_de('combinado')
            sel_c = (w_c >= lo - 60) & (w_c <= hi + 60)
            ax.plot(w_c[sel_c], _razon(w_c, s_c, banda)[sel_c], lw=1.5,
                    color='crimson', label='combinado', zorder=3)
            # Limites de los DATOS: un ylim fijo recorta la banda sin avisar,
            # que es lo que pasaba en la figura equivalente de §6.
            todos = np.concatenate(vistos) if vistos else np.array([np.nan])
            bajo, alto = np.nanpercentile(todos, 0.2), np.nanpercentile(todos, 99.8)
            margen = 0.06 * (alto - bajo)
            ax.set_ylim(bajo - margen, alto + margen)
            ax.tick_params(labelsize=7)
            ax.set_title(f"{nombre} · {'CORREGIDAS por molecfit' if corregir else 'sin corregir'}",
                         fontsize=8.5)
            if fila == 1:
                ax.set_xlabel('λ [Å]', fontsize=8)
    axes[0][0].set_ylabel('flujo / continuo local', fontsize=8)
    axes[1][0].set_ylabel('flujo / continuo local', fontsize=8)
    axes[0][0].legend(fontsize=7, loc='lower left')
    if norma is not None:
        fig.colorbar(plt.cm.ScalarMappable(norm=norma, cmap='viridis'),
                     ax=axes, label='masa de aire (mitad de exposición)')
    plt.show()

    # ¿Se cierra el abanico? Dispersion ENTRE exposiciones, canal a canal.
    print(f'{"banda":<8s}{"disp. sin corregir":>20s}{"disp. corregidas":>18s}'
          f'{"factor":>9s}')
    for nombre, banda in bandas:
        m = None
        pilas = {}
        for corregir in (False, True):
            filas = []
            for e in exps:
                w, s = _corregido(e) if corregir else _espectro_de(e)
                if m is None:
                    m = window_mask(w, banda)
                filas.append(_razon(w, s, banda)[m])
            pilas[corregir] = float(np.nanmedian(np.nanstd(np.vstack(filas), axis=0)))
        antes, despues = pilas[False], pilas[True]
        print(f'{nombre:<8s}{antes:>20.4f}{despues:>18.4f}'
              f'{antes / despues:>9.2f}×')
    print('\n«factor» > 1 = las exposiciones se parecen MÁS tras corregir.')


### 14.4 · La prueba: corregir con cada una y medir lo que queda

Dos maneras de llegar a un espectro corregido del mismo OB:

- **por exposición** — cada espectro se divide por **su propia** transmisión y después se suman, pesando por `EXPTIME`;
- **combinado** — el espectro del cubo ya combinado se divide por la **única** transmisión ajustada sobre él.

Y un **control**: sumar las exposiciones **sin corregir**, que es lo que queda si no se hace nada.

> Sumar espectros no es combinar cubos —el DRS pondera píxel a píxel—, así que esto compara **las dos correcciones**, no las dos reducciones. Para la pregunta de qué transmisión describe mejor la banda, es el contraste que toca.

**La métrica**, la misma que destapó el problema: el **rms del residuo por canal** dentro de la banda, `rms(flujo/continuo − 1)`, contra el **suelo de ruido** del mismo espectro en una ventana limpia (7750–7860 Å, fuera de toda banda ajustada). Un cociente de 1 sería «lo que queda es ruido». La mediana por banda va la última a propósito: es la que cancela el zigzag y la que subestima el error ~6×.


In [ ]:
VENTANA_LIMPIA = (7750.0, 7860.0)   # sin telúrico y fuera de los rangos ajustados

def _residuo(wave, spec, banda):
    """(mediana %, rms por canal) del espectro contra su continuo local."""
    cont = local_continuum_linear(wave, spec, banda,
                                  side_width_A=SIDE_WIDTH_A, gap_A=GAP_A)
    m = window_mask(wave, banda) & np.isfinite(spec) & np.isfinite(cont) & (cont != 0)
    if not m.any():
        return float('nan'), float('nan')
    razon = spec[m] / cont[m]
    return 100.0 * (1.0 - float(np.nanmedian(razon))), float(np.nanstd(razon - 1.0))

if MF is None:
    print('sin ajustes: nada que comparar')
else:
    peso = {f['etiqueta']: float(f['exptime'] or 0.0) for f in MF}
    exps = [f['etiqueta'] for f in MF if f['etiqueta'] != 'combinado']

    acum_corr = acum_sin = None
    total = 0.0
    malla = None
    for e in exps:
        w, s = _espectro_de(e)
        _, corr = _corregido(e)
        if malla is None:
            malla = w
        # Malla comun: si no lo fuera, sumar seria mezclar canales distintos.
        assert np.allclose(w, malla), f'{e}: otra malla de longitud de onda'
        acum_corr = corr * peso[e] if acum_corr is None else acum_corr + corr * peso[e]
        acum_sin = s * peso[e] if acum_sin is None else acum_sin + s * peso[e]
        total += peso[e]
    esp_porexp, esp_sin = acum_corr / total, acum_sin / total

    w_cmb, esp_comb = _corregido('combinado')

    modos = [('sin corregir (control)', malla, esp_sin),
             ('molecfit por exposición', malla, esp_porexp),
             ('molecfit sobre el combinado', w_cmb, esp_comb)]
    print(f'{"modo":<30s}{"banda":<8s}{"mediana %":>11s}{"rms":>9s}'
          f'{"suelo":>9s}{"rms/suelo":>11s}')
    RES = {}
    for etq, w, s in modos:
        _, suelo = _residuo(w, s, VENTANA_LIMPIA)
        for nombre in ('O2_B', 'O2_A'):
            med, rms = _residuo(w, s, BANDAS_MAS[nombre])
            RES[(etq, nombre)] = (med, rms, suelo)
            print(f'{etq:<30s}{nombre:<8s}{med:>11.3f}{rms:>9.4f}'
                  f'{suelo:>9.4f}{rms / suelo:>11.2f}')
    mejor = min(('molecfit por exposición', 'molecfit sobre el combinado'),
                key=lambda k: RES[(k, 'O2_A')][1] / RES[(k, 'O2_A')][2])
    print(f'\nen O₂ A, el rms/suelo más bajo lo da: {mejor}')
    print('(criterio de §5.3 del documento de indicaciones; el desempate, si las dos')
    print(' quedan igual, va a favor de por exposición)')


### 14.5 · Cómo se lee, y qué NO decide esto

- **`rms/suelo` cerca de 1** → lo que queda en la banda es ruido: la corrección ha hecho su trabajo.
- **`rms/suelo` alto con mediana pequeña** → el caso que ya se dio con `STD_TELLURIC`: la banda parece corregida *de media* y está mal canal a canal. La mediana sola no lo ve.
- **La fila «sin corregir»** es la referencia: si una corrección no baja el `rms/suelo` respecto a ella, no está corrigiendo, está añadiendo.

Lo que esta sección **no** decide:

1. **Si la banda residual es telúrica o fotosférica.** Sigue abierto (`docs/2026-08-02_a3_continuo_sesgo.md` §6) y hace falta una plantilla del tipo espectral de la primaria.
2. **Si molecfit puede corregir el cubo entero.** Hoy no: `molecfit_calctrans` está bloqueado en esta build y solo hay transmisión dentro de las ventanas ajustadas.
3. **Si cambia el resultado de la cadena.** Si cambia, aguas abajo hay que re-ejecutar — y eso es una decisión aparte, con su propia aprobación.
